## Data Engineering Frl

In [ ]:
!pip install skimpy mlflow pyngrok xgboost lightgbm econml dowhy causalml shap polars

In [ ]:
import pandas as pd
import polars as pl
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import skimpy as sk
import mlflow
import mlflow.sklearn
import gc
import psutil
import os
from pyngrok import ngrok


from sklearn.preprocessing import *
from sklearn.impute import SimpleImputer
from sklearn.model_selection import *
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')



#Model Export
import joblib

In [ ]:
for files in os.scandir("/content/drive/MyDrive/march-machine-learning-mania-2026"):
  print(files)

<DirEntry 'Cities.csv'>
<DirEntry 'MGameCities.csv'>
<DirEntry 'MConferenceTourneyGames.csv'>
<DirEntry 'Conferences.csv'>
<DirEntry 'MNCAATourneyDetailedResults.csv'>
<DirEntry 'SampleSubmissionStage2.csv'>
<DirEntry 'WNCAATourneyCompactResults.csv'>
<DirEntry 'MNCAATourneyCompactResults.csv'>
<DirEntry 'MSecondaryTourneyTeams.csv'>
<DirEntry 'MNCAATourneySlots.csv'>
<DirEntry 'MTeamSpellings.csv'>
<DirEntry 'WRegularSeasonCompactResults.csv'>
<DirEntry 'MNCAATourneySeedRoundSlots.csv'>
<DirEntry 'MSecondaryTourneyCompactResults.csv'>
<DirEntry 'MMasseyOrdinals.csv'>
<DirEntry 'MNCAATourneySeeds.csv'>
<DirEntry 'MTeams.csv'>
<DirEntry 'WGameCities.csv'>
<DirEntry 'SampleSubmissionStage1.csv'>
<DirEntry 'MTeamConferences.csv'>
<DirEntry 'MTeamCoaches.csv'>
<DirEntry 'MRegularSeasonCompactResults.csv'>
<DirEntry 'MSeasons.csv'>
<DirEntry 'WNCAATourneySeeds.csv'>
<DirEntry 'WRegularSeasonDetailedResults.csv'>
<DirEntry 'MRegularSeasonDetailedResults.csv'>
<DirEntry 'WNCAATourneyDetailedR

In [ ]:
# Memory monitoring function
def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem = process.memory_info().rss / 1024 / 1024 / 1024  # GB
    print(f"Memory usage: {mem:.2f} GB")
    return mem

class MarchMadnessDataGenerator:
    def __init__(self, data_path='/content/drive/MyDrive/march-machine-learning-mania-2026/'):
        self.data_path = data_path
        self.seasons = range(2003, 2025)

    def load_all_data_chunked(self):
        """Load all CSV files with memory-efficient settings"""
        print("Loading all data files with memory optimization...")
        print_memory_usage()

        # Teams data (small, load fully)
        self.teams_m = pl.read_csv(f'{self.data_path}MTeams.csv')
        self.teams_w = pl.read_csv(f'{self.data_path}WTeams.csv')

        # Sample submission (load fully - required)
        self.sample_submission = pl.read_csv(f'{self.data_path}SampleSubmissionStage1.csv')
        print(f"Sample submission: {self.sample_submission.height} rows")

        # Load regular season data with minimal columns and optimizations
        print("\nLoading regular season data...")

        # Men's regular season - only keep needed columns
        self.reg_season_m = pl.read_csv(
            f'{self.data_path}MRegularSeasonCompactResults.csv',
            columns=['Season', 'WTeamID', 'LTeamID', 'WScore', 'LScore'],
            infer_schema_length=1000
        )
        print(f"Men's regular season: {self.reg_season_m.height} rows")

        # Women's regular season
        self.reg_season_w = pl.read_csv(
            f'{self.data_path}WRegularSeasonCompactResults.csv',
            columns=['Season', 'WTeamID', 'LTeamID', 'WScore', 'LScore'],
            infer_schema_length=1000
        )
        print(f"Women's regular season: {self.reg_season_w.height} rows")

        # Tournament data
        print("\nLoading tournament data...")
        self.tourney_m = pl.read_csv(f'{self.data_path}MNCAATourneyCompactResults.csv')
        self.tourney_w = pl.read_csv(f'{self.data_path}WNCAATourneyCompactResults.csv')

        # Seeds
        self.seeds_m = pl.read_csv(f'{self.data_path}MNCAATourneySeeds.csv')
        self.seeds_w = pl.read_csv(f'{self.data_path}WNCAATourneySeeds.csv')

        # Team conferences (small)
        self.team_conferences_m = pl.read_csv(f'{self.data_path}MTeamConferences.csv')
        self.team_conferences_w = pl.read_csv(f'{self.data_path}WTeamConferences.csv')

        # Massey Ordinals - load only what we need
        print("\nLoading Massey Ordinals...")
        try:
            # Check file size first
            file_size = os.path.getsize(f'{self.data_path}MMasseyOrdinals.csv') / (1024**3)
            if file_size > 1:  # If > 1GB
                print(f"Large Massey file ({file_size:.2f} GB), loading with chunking...")
                # Load only needed columns and systems
                self.massey_m = pl.read_csv(
                    f'{self.data_path}MMasseyOrdinals.csv',
                    columns=['Season', 'TeamID', 'SysName', 'OrdinalRank'],
                    infer_schema_length=1000
                )
                # Filter to only common systems immediately
                self.massey_m = self.massey_m.filter(
                    pl.col('SysName').is_in(['POM', 'SAG', 'MOR', 'WLK'])
                )
            else:
                self.massey_m = pl.read_csv(f'{self.data_path}MMasseyOrdinals.csv')

            print(f"Men's Massey: {self.massey_m.height} rows")

            # Women's Massey
            self.massey_w = pl.read_csv(
                f'{self.data_path}WMasseyOrdinals.csv',
                columns=['Season', 'TeamID', 'SysName', 'OrdinalRank'],
                infer_schema_length=1000
            )
            self.massey_w = self.massey_w.filter(
                pl.col('SysName').is_in(['POM', 'SAG', 'MOR', 'WLK'])
            )
            print(f"Women's Massey: {self.massey_w.height} rows")

        except Exception as e:
            print(f"Massey loading issue: {e}")
            self.massey_m = None
            self.massey_w = None

        print("\nAll data loaded successfully!")
        print_memory_usage()
        gc.collect()

    def extract_seed_number(self, seed_expr):
        """Extract numeric seed from seed string"""
        return (
            pl.when(seed_expr.is_null())
            .then(16)
            .otherwise(seed_expr.str.slice(1, 2).cast(pl.Int32))
        )

    def create_team_features_lazy(self, gender='M'):
        """Create team features using lazy evaluation"""
        print(f"Creating {gender} team features (lazy)...")

        if gender == 'M':
            reg_season = self.reg_season_m.lazy()
            teams = self.teams_m.lazy()
            team_conferences = self.team_conferences_m.lazy()
            seeds = self.seeds_m.lazy()
            massey = self.massey_m.lazy() if self.massey_m is not None else None
        else:
            reg_season = self.reg_season_w.lazy()
            teams = self.teams_w.lazy()
            team_conferences = self.team_conferences_w.lazy()
            seeds = self.seeds_w.lazy()
            massey = self.massey_w.lazy() if self.massey_w is not None else None

        # Wins
        wins = reg_season.group_by(['Season', 'WTeamID']).agg([
            pl.count().alias('Wins'),
            pl.sum('WScore').alias('PointsFor_Wins'),
            pl.sum('LScore').alias('PointsAgainst_Wins')
        ]).rename({'WTeamID': 'TeamID'})

        # Losses
        losses = reg_season.group_by(['Season', 'LTeamID']).agg([
            pl.count().alias('Losses'),
            pl.sum('LScore').alias('PointsFor_Losses'),
            pl.sum('WScore').alias('PointsAgainst_Losses')
        ]).rename({'LTeamID': 'TeamID'})

        # Combine
        team_stats = wins.join(losses, on=['Season', 'TeamID'], how='outer_coalesce').fill_null(0)

        # Calculate stats
        team_stats = team_stats.with_columns([
            (pl.col('Wins') + pl.col('Losses')).alias('NumGames'),
            (pl.col('PointsFor_Wins') + pl.col('PointsFor_Losses')).alias('PointsFor'),
            (pl.col('PointsAgainst_Wins') + pl.col('PointsAgainst_Losses')).alias('PointsAgainst')
        ]).with_columns([
            (pl.col('Wins') / pl.col('NumGames')).alias('WinPct'),
            (pl.col('PointsFor') / pl.col('NumGames')).alias('AvgPointsFor'),
            (pl.col('PointsAgainst') / pl.col('NumGames')).alias('AvgPointsAgainst'),
            ((pl.col('PointsFor') - pl.col('PointsAgainst')) / pl.col('NumGames')).alias('PointDiff')
        ])

        # Add conferences
        team_stats = team_stats.join(team_conferences, on=['Season', 'TeamID'], how='left')

        # Add seeds
        seeds_processed = seeds.with_columns([
            self.extract_seed_number(pl.col('Seed')).alias('SeedNum')
        ]).select(['Season', 'TeamID', 'SeedNum'])

        team_stats = team_stats.join(seeds_processed, on=['Season', 'TeamID'], how='left')

        # Add team names
        team_stats = team_stats.join(teams.select(['TeamID', 'TeamName']), on='TeamID', how='left')

        # Add Massey ratings (if available)
        if massey is not None:
            # Pivot Massey ratings
            for system in ['POM', 'SAG', 'MOR', 'WLK']:
                system_ratings = massey.filter(pl.col('SysName') == system).select(
                    ['Season', 'TeamID', 'OrdinalRank']
                ).rename({'OrdinalRank': f'Massey_{system}'})

                team_stats = team_stats.join(system_ratings, on=['Season', 'TeamID'], how='left')

        # Collect with optimization
        result = team_stats.collect()
        print(f"Created {result.height} {gender} team-season records")
        print_memory_usage()

        return result

    def generate_test_data_chunked(self):
        """Generate test data in chunks to save memory"""
        print("\n" + "="*60)
        print("GENERATING TEST DATA IN CHUNKS")
        print("="*60)

        # Create team features for lookups
        print("\nCreating team features for lookups...")
        team_stats_m = self.create_team_features_lazy('M')
        team_stats_w = self.create_team_features_lazy('W')

        # Convert to pandas for easier lookup (but only keep essential columns)
        print("\nPreparing lookup dictionaries...")

        # Helper function to safely get value
        def safe_get(row, col, default):
            val = row.get(col)
            if val is None or pd.isna(val):
                return default
            return val

        # For men
        team_dict_m = {}
        for row in team_stats_m.iter_rows(named=True):
            key = (row['Season'], row['TeamID'])
            team_dict_m[key] = {
                'TeamName': safe_get(row, 'TeamName', f"Team_{row['TeamID']}"),
                'WinPct': safe_get(row, 'WinPct', 0.5),
                'AvgPointsFor': safe_get(row, 'AvgPointsFor', 70.0),
                'AvgPointsAgainst': safe_get(row, 'AvgPointsAgainst', 70.0),
                'PointDiff': safe_get(row, 'PointDiff', 0.0),
                'NumGames': safe_get(row, 'NumGames', 30),
                'SeedNum': safe_get(row, 'SeedNum', 16),
                'ConfAbbrev': safe_get(row, 'ConfAbbrev', 'UNK'),
                'Massey_POM': safe_get(row, 'Massey_POM', np.nan),
                'Massey_SAG': safe_get(row, 'Massey_SAG', np.nan),
                'Massey_MOR': safe_get(row, 'Massey_MOR', np.nan),
                'Massey_WLK': safe_get(row, 'Massey_WLK', np.nan)
            }

        # For women
        team_dict_w = {}
        for row in team_stats_w.iter_rows(named=True):
            key = (row['Season'], row['TeamID'])
            team_dict_w[key] = {
                'TeamName': safe_get(row, 'TeamName', f"Team_{row['TeamID']}"),
                'WinPct': safe_get(row, 'WinPct', 0.5),
                'AvgPointsFor': safe_get(row, 'AvgPointsFor', 70.0),
                'AvgPointsAgainst': safe_get(row, 'AvgPointsAgainst', 70.0),
                'PointDiff': safe_get(row, 'PointDiff', 0.0),
                'NumGames': safe_get(row, 'NumGames', 30),
                'SeedNum': safe_get(row, 'SeedNum', 16),
                'ConfAbbrev': safe_get(row, 'ConfAbbrev', 'UNK'),
                'Massey_POM': safe_get(row, 'Massey_POM', np.nan),
                'Massey_SAG': safe_get(row, 'Massey_SAG', np.nan),
                'Massey_MOR': safe_get(row, 'Massey_MOR', np.nan),
                'Massey_WLK': safe_get(row, 'Massey_WLK', np.nan)
            }

        # Free memory
        del team_stats_m
        del team_stats_w
        gc.collect()
        print_memory_usage()

        # Process sample submission in chunks
        sample_pd = self.sample_submission.to_pandas()
        total_rows = len(sample_pd)
        chunk_size = 25000  # Reduced chunk size for safety

        men_chunks = []
        women_chunks = []

        print(f"\nProcessing {total_rows} rows in chunks of {chunk_size}...")

        for start_idx in range(0, total_rows, chunk_size):
            end_idx = min(start_idx + chunk_size, total_rows)
            print(f"\nProcessing chunk {start_idx//chunk_size + 1}: rows {start_idx}-{end_idx}")

            chunk = sample_pd.iloc[start_idx:end_idx].copy()

            # Split chunk into men/women
            men_rows = []
            women_rows = []

            for _, row in chunk.iterrows():
                id_parts = row['ID'].split('_')
                if len(id_parts) != 3:
                    continue

                team1 = int(id_parts[1])

                if team1 < 2000:
                    men_rows.append(row)
                else:
                    women_rows.append(row)

            # Process men's chunk
            if men_rows:
                men_chunk_df = pd.DataFrame(men_rows)
                men_processed = self._process_chunk(men_chunk_df, team_dict_m, 'M')
                if men_processed is not None and len(men_processed) > 0:
                    men_chunks.append(men_processed)

            # Process women's chunk
            if women_rows:
                women_chunk_df = pd.DataFrame(women_rows)
                women_processed = self._process_chunk(women_chunk_df, team_dict_w, 'W')
                if women_processed is not None and len(women_processed) > 0:
                    women_chunks.append(women_processed)

            # Clear chunk from memory
            del chunk
            del men_rows
            del women_rows
            gc.collect()
            print_memory_usage()

        # Combine chunks
        print("\nCombining chunks...")
        men_test = pd.concat(men_chunks, ignore_index=True) if men_chunks else pd.DataFrame()
        women_test = pd.concat(women_chunks, ignore_index=True) if women_chunks else pd.DataFrame()

        # Verify total
        total = len(men_test) + len(women_test)
        print("\n" + "="*60)
        print(f"VERIFICATION:")
        print(f"  Men's test rows: {len(men_test)}")
        print(f"  Women's test rows: {len(women_test)}")
        print(f"  COMBINED TOTAL: {total}")
        print(f"  Target: {total_rows}")
        print(f"  Match: {total == total_rows}")
        print("="*60)

        return men_test, women_test

    def _process_chunk(self, chunk_df, team_dict, gender='M'):
        """Process a single chunk of test data"""
        if len(chunk_df) == 0:
            return None

        # Define feature columns
        base_cols = [
            'Season', 'Team1ID', 'Team2ID', 'Team1Name', 'Team2Name',
            'Team1_WinPct', 'Team1_AvgPointsFor', 'Team1_AvgPointsAgainst',
            'Team1_PointDiff', 'Team1_NumGames', 'Team1_Seed',
            'Team2_WinPct', 'Team2_AvgPointsFor', 'Team2_AvgPointsAgainst',
            'Team2_PointDiff', 'Team2_NumGames', 'Team2_Seed',
            'H2H_Team1Wins', 'H2H_Team2Wins',
            'Diff_WinPct', 'Diff_PointDiff', 'Diff_Seed',
            'SameConference'
        ]

        # Add Massey columns
        massey_cols = []
        for system in ['POM', 'SAG', 'MOR', 'WLK']:
            massey_cols.extend([
                f'Team1_Massey_{system}',
                f'Team2_Massey_{system}',
                f'Diff_Massey_{system}'
            ])

        # Initialize result dataframe
        result = chunk_df.copy()
        for col in base_cols + massey_cols:
            result[col] = np.nan

        # Get regular season for H2H
        if gender == 'M':
            reg_season = self.reg_season_m
        else:
            reg_season = self.reg_season_w

        # Create H2H lookup for this chunk's seasons
        seasons_in_chunk = set()
        for _, row in chunk_df.iterrows():
            id_parts = row['ID'].split('_')
            if len(id_parts) == 3:
                seasons_in_chunk.add(int(id_parts[0]))

        # Pre-filter H2H data for these seasons
        h2h_filtered = reg_season.filter(pl.col('Season').is_in(list(seasons_in_chunk)))
        h2h_pd = h2h_filtered.to_pandas()

        # Create H2H lookup dictionary
        h2h_dict = {}
        for _, game in h2h_pd.iterrows():
            season = game['Season']
            wteam = game['WTeamID']
            lteam = game['LTeamID']
            key1 = (season, wteam, lteam)
            key2 = (season, lteam, wteam)
            h2h_dict[key1] = (1, 0)  # team1 wins, team2 loses
            h2h_dict[key2] = (0, 1)  # team2 wins, team1 loses

        # Process each row
        for idx, row in result.iterrows():
            if idx % 5000 == 0 and idx > 0:
                print(f"    Processed {idx}/{len(result)} rows in chunk")

            id_parts = row['ID'].split('_')
            if len(id_parts) != 3:
                continue

            season = int(id_parts[0])
            team1 = int(id_parts[1])
            team2 = int(id_parts[2])

            # Get team stats - try multiple seasons if needed
            t1 = None
            t2 = None

            # Try current season
            key1 = (season, team1)
            key2 = (season, team2)

            if key1 in team_dict:
                t1 = team_dict[key1]
            else:
                # Try previous season
                key1_prev = (season-1, team1)
                if key1_prev in team_dict:
                    t1 = team_dict[key1_prev]

            if key2 in team_dict:
                t2 = team_dict[key2]
            else:
                key2_prev = (season-1, team2)
                if key2_prev in team_dict:
                    t2 = team_dict[key2_prev]

            # Skip if still missing
            if t1 is None or t2 is None:
                continue

            # Get H2H
            h2h_key = (season, team1, team2)
            h2h_result = h2h_dict.get(h2h_key, (0, 0))

            # Fill features with safe operations
            result.at[idx, 'Season'] = season
            result.at[idx, 'Team1ID'] = team1
            result.at[idx, 'Team2ID'] = team2
            result.at[idx, 'Team1Name'] = t1.get('TeamName', f'Team_{team1}')
            result.at[idx, 'Team2Name'] = t2.get('TeamName', f'Team_{team2}')

            # Team 1 - ensure numeric values
            result.at[idx, 'Team1_WinPct'] = float(t1.get('WinPct', 0.5))
            result.at[idx, 'Team1_AvgPointsFor'] = float(t1.get('AvgPointsFor', 70.0))
            result.at[idx, 'Team1_AvgPointsAgainst'] = float(t1.get('AvgPointsAgainst', 70.0))
            result.at[idx, 'Team1_PointDiff'] = float(t1.get('PointDiff', 0.0))
            result.at[idx, 'Team1_NumGames'] = int(t1.get('NumGames', 30))
            result.at[idx, 'Team1_Seed'] = int(t1.get('SeedNum', 16))

            # Team 2 - ensure numeric values
            result.at[idx, 'Team2_WinPct'] = float(t2.get('WinPct', 0.5))
            result.at[idx, 'Team2_AvgPointsFor'] = float(t2.get('AvgPointsFor', 70.0))
            result.at[idx, 'Team2_AvgPointsAgainst'] = float(t2.get('AvgPointsAgainst', 70.0))
            result.at[idx, 'Team2_PointDiff'] = float(t2.get('PointDiff', 0.0))
            result.at[idx, 'Team2_NumGames'] = int(t2.get('NumGames', 30))
            result.at[idx, 'Team2_Seed'] = int(t2.get('SeedNum', 16))

            # H2H
            result.at[idx, 'H2H_Team1Wins'] = int(h2h_result[0])
            result.at[idx, 'H2H_Team2Wins'] = int(h2h_result[1])

            # Differences - ensure both are numeric
            t1_winpct = float(t1.get('WinPct', 0.5))
            t2_winpct = float(t2.get('WinPct', 0.5))
            t1_pointdiff = float(t1.get('PointDiff', 0.0))
            t2_pointdiff = float(t2.get('PointDiff', 0.0))
            t1_seed = int(t1.get('SeedNum', 16))
            t2_seed = int(t2.get('SeedNum', 16))

            result.at[idx, 'Diff_WinPct'] = t1_winpct - t2_winpct
            result.at[idx, 'Diff_PointDiff'] = t1_pointdiff - t2_pointdiff
            result.at[idx, 'Diff_Seed'] = t2_seed - t1_seed

            # Same conference
            conf1 = t1.get('ConfAbbrev', 'UNK')
            conf2 = t2.get('ConfAbbrev', 'UNK')
            result.at[idx, 'SameConference'] = 1 if conf1 == conf2 else 0

            # Massey
            for system in ['POM', 'SAG', 'MOR', 'WLK']:
                m1 = t1.get(f'Massey_{system}', np.nan)
                m2 = t2.get(f'Massey_{system}', np.nan)

                # Convert to float if not None/NaN
                if m1 is None or pd.isna(m1):
                    m1 = np.nan
                if m2 is None or pd.isna(m2):
                    m2 = np.nan

                result.at[idx, f'Team1_Massey_{system}'] = m1
                result.at[idx, f'Team2_Massey_{system}'] = m2

                if not pd.isna(m1) and not pd.isna(m2):
                    result.at[idx, f'Diff_Massey_{system}'] = float(m2) - float(m1)

        # Fill NaNs with medians
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col not in ['Pred'] and col in result.columns:
                # Calculate median safely
                if result[col].notna().any():
                    median_val = result[col].median()
                else:
                    median_val = 0
                result[col] = result[col].fillna(median_val)

        return result

    def run(self):
        """Main execution method"""
        print("="*60)
        print("MARCH MACHINE LEARNING MANIA - MEMORY OPTIMIZED GENERATION")
        print("="*60)

        # Load data with memory optimization
        self.load_all_data_chunked()

        # Generate test data in chunks
        men_test, women_test = self.generate_test_data_chunked()

        # Save results
        print("\nSaving results...")
        if len(men_test) > 0:
            men_test.to_csv('men_test_data_1.csv', index=False)
            print(f"Men's test data saved: {len(men_test)} rows")

        if len(women_test) > 0:
            women_test.to_csv('women_test_data_1.csv', index=False)
            print(f"Women's test data saved: {len(women_test)} rows")

        print("\n" + "="*60)
        print("PROCESSING COMPLETE!")
        print("="*60)
        print_memory_usage()

        return men_test, women_test

# Run with memory monitoring
if __name__ == "__main__":
    # Enable garbage collection
    gc.enable()

    # Create generator and run
    generator = MarchMadnessDataGenerator()
    men_test, women_test = generator.run()

    # Show sample
    if len(men_test) > 0:
        print("\nMen's test sample:")
        display_cols = ['ID', 'Team1Name', 'Team2Name', 'Team1_Seed', 'Team2_Seed']
        available_cols = [c for c in display_cols if c in men_test.columns]
        print(men_test[available_cols].head())

    if len(women_test) > 0:
        print("\nWomen's test sample:")
        display_cols = ['ID', 'Team1Name', 'Team2Name', 'Team1_Seed', 'Team2_Seed']
        available_cols = [c for c in display_cols if c in women_test.columns]
        print(women_test[available_cols].head())

MARCH MACHINE LEARNING MANIA - MEMORY OPTIMIZED GENERATION
Loading all data files with memory optimization...
Memory usage: 1.42 GB
Sample submission: 519144 rows

Loading regular season data...
Men's regular season: 196823 rows
Women's regular season: 140825 rows

Loading tournament data...

Loading Massey Ordinals...
Men's Massey: 5761702 rows
Massey loading issue: No such file or directory (os error 2): /content/drive/MyDrive/march-machine-learning-mania-2026/WMasseyOrdinals.csv

All data loaded successfully!
Memory usage: 1.52 GB

GENERATING TEST DATA IN CHUNKS

Creating team features for lookups...
Creating M team features (lazy)...
Created 13753 M team-season records
Memory usage: 1.44 GB
Creating W team features (lazy)...
Created 9851 W team-season records
Memory usage: 1.44 GB

Preparing lookup dictionaries...
Memory usage: 1.44 GB

Processing 519144 rows in chunks of 25000...

Processing chunk 1: rows 0-25000
    Processed 5000/25000 rows in chunk
    Processed 10000/25000 row

In [ ]:
!zip train_and_test_files.zip men_training_data.csv women_training_data.csv men_test_data.csv women_test_data.csv